### Evaluate interpretations from the model

In [ ]:
%load_ext autoreload
%autoreload 2

# Run in parent dir
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from dual_ifm.classification.eval_finetune import get_all_metrics, load_eval
from dual_ifm.interpretation import eval
from dual_ifm.utils import bagnetsv2 as bagnets
from dual_ifm.utils import datasets, plot

# Run in parent dir
cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

plot.set_rc_params(kind='paper', notebook_dpi=120)
pd.options.display.float_format = '{:,.3f}'.format
device = 'cuda:0'

In [ ]:
dataset_dir = './datasets'
prefix = 'tsimcneb'
dataset_name = 'idrid'
feature_name = 'drbin'
kfold = 0
img_size = (256, 256)  # Size of the input to the model
sweep_name = f'sweep_{dataset_name}_sparsity'
experiment_name = 'model.sparsity=0.pt'

project_dir = Path.cwd()
checkpoints_dir = project_dir.joinpath('checkpoints')

### Class evidence maps

In [ ]:
# Load dataset
normalization = {'mean': datasets.NORMALIZATION_MEAN[dataset_name], 'sd': datasets.NORMALIZATION_SD[dataset_name]}
transform = datasets.get_augmentations(img_size=img_size, normalization=normalization, imagenet=False)['test']
dataset, mapping = datasets.load_dataset(dataset_dir=dataset_dir, dataset_name=dataset_name, transform=None, image_size=img_size, feature_name=feature_name, drop_nan=True, sample_size=None, split='all')
n_classes = len(mapping)

# Load model
model = eval.load_model(checkpoints_dir / sweep_name / experiment_name, img_size, n_classes, device)

# To plot the heatmaps
circle_mask = cv2.circle(np.zeros(img_size), (img_size[0]//2, img_size[1]//2), img_size[0]//2, np.nan, -1)
circle_mask[:8, :] = 0

In [ ]:
transform_original = transform.transforms[0]
idx = 0

image, label = dataset[idx]
with torch.no_grad():
    img = transform(image).unsqueeze(0).to(device)
    heatmap = model(img, return_heatmap=True).squeeze().cpu().numpy()

In [ ]:
fig, ax = plt.subplots(1, 2)
plot.set_figsize(fig, 'col', height_ratio=0.5)
ax[0].imshow(image)
ax[0].set_title(label)
ax[0].axis("off")
bagnets.plot_heatmap(heatmap[int(label),:,:], transform_original(image), fig, ax[1], percentile=100, mask=circle_mask)
plot.tight_layout()

### Compare models with sparsity

In [ ]:
sparsity_lambdas = [0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]

df_rows = []
for sparsity in sparsity_lambdas:
    # Experiment name
    experiment_name = f'model.sparsity={sparsity}'

    if os.path.exists(checkpoints_dir.joinpath(sweep_name, experiment_name + '.json')):
        preds, probs, targets, _, _ = load_eval(checkpoints_dir.joinpath(sweep_name), experiment_name)
        auroc, auprc, acc, kappa = get_all_metrics(preds, probs, targets)
        row = {
            'Sparsity': sparsity,
            'AUROC': auroc,
            'AUPRC': auprc,
            'Balanced Accuracy': acc,
            'Kappa': kappa,
        }
        df_rows.append(row)

df = pd.DataFrame(df_rows)

In [ ]:
df

In [ ]:
fig, ax = plt.subplots(1, 1)
plot.set_figsize(fig, 'col', height_ratio=0.8)
sns.lineplot(df, x='Sparsity', y='AUROC', label='AUROC', marker='o')
sns.lineplot(df, x='Sparsity', y='AUPRC', label='AUPRC', marker='o')
sns.scatterplot(x=[1e-4], y=[0.865], color='green', s=20, zorder=5, label='Selected model')
sns.scatterplot(x=[1e-4], y=[0.946], color='green', s=20, zorder=5)
ax.set_xscale('log')
ax.set_ylabel('Performance')
plot.tight_layout()

### Evaluation with segmentation masks

In [ ]:
# Load IDRiD with annotations
dataset_name = 'idridncc'
normalization = {'mean': datasets.NORMALIZATION_MEAN[dataset_name], 'sd': datasets.NORMALIZATION_SD[dataset_name]}
transform = datasets.get_augmentations(img_size=img_size, normalization=normalization, imagenet=False)['test']
dataset = datasets.IDRiDSegmentation(os.path.join(dataset_dir, dataset_name), transform=None)
topk = 10
n_patches = img_size[0] // 32

In [ ]:
scores = []
for sparsity in sparsity_lambdas:
    # Experiment name
    experiment_name = f'model.sparsity={sparsity}'
    checkpoint_file = checkpoints_dir.joinpath(sweep_name, experiment_name + '.pt')

    if os.path.exists(checkpoint_file):
        model = eval.load_model(checkpoint_file, img_size, n_classes, device)
        score = eval.get_dataset_score(model, dataset, transform, img_size, n_patches, topk=topk, device='cuda:0')
        scores.append(score)

df['Precision'] = scores

In [ ]:
scores

In [ ]:
fig, ax = plt.subplots(1, 1)
plot.set_figsize(fig, 'col', height_ratio=0.8)
sns.lineplot(df, x='Sparsity', y='Precision', marker='o')
sns.scatterplot(x=[1e-4], y=[0.666], color='green', s=20, zorder=5, label='Selected model')
ax.set_xscale('log')
ax.set_ylabel('Precision (k=10)')
plot.tight_layout()

### Interpretability figure

In [ ]:
images, masks, heatmaps = [], [], []
dataset_idxs = [38, 55]

checkpoint_file = checkpoints_dir.joinpath(sweep_name, f'model.sparsity={1e-4}.pt')
model = eval.load_model(checkpoint_file, img_size, n_classes, device)

# For idridncc
circle_mask = cv2.circle(np.zeros(img_size), (img_size[0]//2, img_size[1]//2), img_size[0]//2, np.nan, -1)
circle_mask[:20, :] = 0
circle_mask[-19:, :] = 0

for idx in dataset_idxs:
    image, mask, label = dataset[idx]

    with torch.no_grad():
        img = transform(image).unsqueeze(0).to(device)
        heatmap = model(img, return_heatmap=True).squeeze().cpu().numpy()
        mask = np.array(mask.convert('1'), dtype=float)
        mask = cv2.resize(mask, dsize=img_size, interpolation=cv2.INTER_NEAREST)

    images.append(image)
    masks.append(mask)
    heatmaps.append(heatmap[label, :, :])

In [ ]:
fig, ax = plt.subplots(2, 3)
plot.set_figsize(fig, 'full', height_ratio=0.67)


for i, (image, mask, heatmap) in enumerate(zip(images, masks, heatmaps)):    
    # Image
    ax[i, 1].imshow(image)
    ax[i, 1].axis("off")

    # Heatmap with lesions
    bagnets.plot_heatmap(heatmap, transform_original(image), fig, ax[i, 2], percentile=99, mask=circle_mask)
    mask_plot = mask
    mask_plot[mask < 1] = np.nan
    ax[i, 2].imshow(mask_plot, cmap='winter_r')

# AUROC, AUPRC curve
sns.lineplot(df, x='Sparsity', y='AUROC', label='AUROC', marker='o', color='k', ax=ax[0, 0])
sns.lineplot(df, x='Sparsity', y='AUPRC', label='AUPRC', marker='o', color='gray', ax=ax[0, 0])
sns.scatterplot(x=[1e-4], y=[0.946], color='green', s=20, zorder=5, label='Selected', ax=ax[0, 0])
sns.scatterplot(x=[1e-4], y=[0.865], color='green', s=20, zorder=5, ax=ax[0, 0])
ax[0, 0].set_xscale('log')
ax[0, 0].set_ylabel('Performance')
ax[0, 0].set_ylim([0.5, 1])
plot.dettach_axes(ax[0, 0])

# Precision curve
sns.lineplot(df, x='Sparsity', y='Precision', marker='o', color='k', ax=ax[1, 0])
sns.scatterplot(x=[1e-4], y=[0.666], color='green', s=20, zorder=5, label='Selected', ax=ax[1, 0])
ax[1, 0].axhline(y=0.384, color='red', linestyle='--', label='RETFound')
ax[1, 0].legend()
ax[1, 0].set_xscale('log')
ax[1, 0].set_ylabel('Precision (k=10)')
ax[1, 0].set_ylim([0.3, 0.7])
plot.dettach_axes(ax[1, 0])

plot.set_labs(ax, panel_nums=['A', 'C', 'D', 'B', 'E', 'F'], panel_num_pad=10)
plot.tight_layout(w_pad=0)

fig.savefig('./plots/local_interpretability.pdf')
fig.savefig('./plots/local_interpretability.svg')
# fig.savefig('plots/measurements.png', bbox_inches='tight')